# PolicyProof AI — prepare blinded human reviews (CPU only)

**This notebook does not train, evaluate, or run any AI model.** It validates the exact saved 48-case paired evaluation ZIP, prepares a randomized A/B human-review CSV, and separately saves a confidential model-identity mapping. All evidence is synthetic development data that has already been inspected.

**Requirements:** The original file `policyproof_dev48_paired_gpu_results.zip` and internet access to clone the public code. The original audited ZIP SHA-256 is prefilled. Don't upload any private Hugging Face token or change the checkpoint. Use **Runtime → Change runtime type → CPU**, not GPU.

**Important:** Download **both** generated files before closing Colab; do not show the private model-mapping JSON to any reviewer. No human review will be performed automatically.
**Privacy hardening:** This version uses opaque random case identifiers and a neutral `review_case` category. Real UNKNOWN/INJECT categories are known **only to the private mapping**, never to the reviewer packet.


## 1 — Clone the current checked-in source (no GPU)

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

work = Path("/content")
repo = work / "PolicyProof-AI"
if not repo.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git", str(repo)],
        check=True,
    )
assert (repo / "scripts/export_review_from_bundle.py").is_file()
print("Repository available:", repo)
print("No model download, training, or inference performed.")

## 2 — Upload the original paired GPU results ZIP

Upload **only** the original `policyproof_dev48_paired_gpu_results.zip`. Do not use the adapter ZIP in this step; this notebook reviews the two saved prediction reports.

In [ ]:
from google.colab import files

os.chdir(work)
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Please upload exactly one paired-results ZIP.")
bundle = (work / zip_names[0]).resolve()
assert bundle.is_file()
print("Uploaded ZIP:", bundle.name, "bytes:", bundle.stat().st_size)

## 3 — Verify ZIP hash, four archive members and report consistency

The fixed SHA protects against accidentally using a different archive. A failure means **stop**; do not substitute a new SHA without independently verifying the actual archive.

In [ ]:
EXPECTED_SHA256 = "e98b06482985a9c2f5dfa62100dccda7421830de8292bae013250ccf4ca1099c"
subprocess.run([sys.executable, "-m", "scripts.prepare"], cwd=repo, check=True)
subprocess.run([
    sys.executable, "-m", "scripts.export_review_from_bundle",
    "--bundle", str(bundle),
    "--expected-sha256", EXPECTED_SHA256,
    "--verify-only",
], cwd=repo, check=True)
print("Verified bundle and frozen dataset; no reviewer file generated yet.")

## 4 — Export a fresh masked, randomized review packet and confidential mapping

This creates `reviews/dev48_blank.csv` and `reviews/dev48_private_mapping.json` exactly once. The CSV contains **no original `UNKNOWN` / `INJECT` IDs, gold answerability categories, model identities, or gold answers**. Every sample has a random `CASE-…` ID and neutral category `review_case`. The JSON mapping contains original categories and model identities; never share that key with a reviewer.

If you used an older v1 packet, do **not** overwrite it or mix it with v2 reviewer files. Keep already completed reviews as a separate historical version, and use a fresh runtime/packet for a new review.


In [ ]:
subprocess.run([
    sys.executable, "-m", "scripts.export_review_from_bundle",
    "--bundle", str(bundle),
    "--expected-sha256", EXPECTED_SHA256,
], cwd=repo, check=True)
packet = repo / "reviews" / "dev48_blank.csv"
private_key = repo / "reviews" / "dev48_private_mapping.json"
assert packet.is_file() and private_key.is_file()
import csv
import re
with packet.open("r", encoding="utf-8-sig", newline="") as fp:
    reviewer_rows = list(csv.DictReader(fp))
assert len(reviewer_rows) == 96
assert {r["category"] for r in reviewer_rows} == {"review_case"}
assert len({r["case_id"] for r in reviewer_rows}) == 48
assert all(re.fullmatch(r"CASE-[0-9a-f]{16}", r["case_id"]) for r in reviewer_rows)
assert all(r["reviewer_code"] == "" for r in reviewer_rows)
private_info = __import__("json").loads(private_key.read_text(encoding="utf-8"))
assert private_info["schema"] == "policyproof_dev48_blinded_semantic_review_v2"
assert private_info["case_ids_masked"] is True
assert private_info["category_masked"] is True
assert len(private_info["case_metadata"]) == 48
print("Verified 96 masked reviewer rows and separate owner-only mapping; no human review performed.")

## 5 — Download BOTH files, then arrange genuinely independent reviews

Share only the **masked, blank CSV** with two competent independent reviewers, covering English, Norwegian and Arabic. Keep the **private mapping JSON** confidential and safely backed up; do not send it to reviewers or commit it to GitHub. Each reviewer should fill their own CSV with three judgments per response. Use the repo guide `docs/DEV48_BLINDED_SEMANTIC_REVIEW.md` to summarize votes later. **No approvals or factual accuracy scores are created by this notebook.**

In [ ]:
files.download(str(packet))
files.download(str(private_key))
print("Keep the private key away from the reviewers. No human semantic review has occurred.")